## Atividade 4 - Chatbot com LLM baseado em Planejamento de Consultas Estruturadas

Esta Atividade 4 dá continuidade ao trabalho iniciado na Atividade 3, na qual foi explorada uma abordagem baseada em RAG com embeddings e recuperação por similaridade semântica. Embora funcional, a solução anterior apresentou limitações importantes ao lidar com dados fortemente estruturados, especialmente em perguntas que exigiam:

* filtros exatos (semestre, professor, modalidade),

* agregações (soma de carga horária, contagens),

* e consistência institucional (não inventar informações).

Diante dessas limitações, nesta atividade o uso de RAG e embeddings foi removido, e o modelo de linguagem passou a ser utilizado em dois pontos bem definidos do pipeline, com funções claramente delimitadas.

<br>

### Mudança de Abordagem em Relação à Atividade 3

| Atividade 3                             | Atividade 4                                     |
| --------------------------------------- | ----------------------------------------------- |
| RAG com embeddings e `top-k`            | Sem RAG e sem embeddings                        |
| Recuperação aproximada de contexto      | Consultas determinísticas em dados estruturados |
| LLM acessa diretamente trechos de dados | LLM não acessa dados brutos                     |
| Dificuldade com agregações              | Agregações corretas e rastreáveis               |


<br>

Na Atividade 4, o modelo de linguagem não executa consultas nem “decide” resultados. Ele atua apenas como:

1. Planejador de consultas, e

2. Redator da resposta final, sempre com base em resultados estruturados.

<br>

### Arquitetura da Solução

1. Dados Estruturados  
Entrada: arquivos JSON institucionais (matriz, horários, calendário)  
Saída: DataFrames persistidos no Google Drive  
Os dados são processados uma única vez e reutilizados por meio de cache.

2. LLM Planner (Planejamento)  
  Entrada:
    * pergunta do usuário (texto livre)  
    * metadados dos datasets  

    Saída:  
    * plano de consulta estruturado (JSON), indicando:  
        * dataset a ser usado,  
        * filtros,  
        * seleções,
        * agregações e agrupamentos.  

    O LLM atua apenas no planejamento, sem acessar os dados.

3. Query Executor (Determinístico)  
  Entrada:
    * plano de consulta (JSON)
    * DataFrames estruturados  

    Saída:

    * resultado estruturado (DataFrame),  
    * diagnóstico da execução.

    Toda a consulta é executada de forma determinística, sem uso de LLM.

4. LLM Summarizer (Geração da Resposta)
    Entrada:  
    * pergunta original,  
    * resultado estruturado da consulta.  

    Saída:
    * resposta textual final ao usuário.

    Nesta etapa, o LLM é utilizado apenas para redigir a resposta, seguindo regras explícitas de não inferência e não invenção de dados.

<br>

Para rodar os exercícios você precisará fornecer o caminho dos arquivos JSON na variável `PATH_JSONS = "caminho/arquivos` na Seção 5:

* Calendário acadêmico (`calendario-2026.json`);  
* Horários das disciplinas (`dsm-horario-2026-1.json`);  
* Matriz curricular (`dsm-matriz-curricular.json`).

In [ ]:
# @title 1.1. Intalação e importações
!pip install -q openai pandas pyarrow jsonschema

import os
import json
import re
import unicodedata
import jsonschema
import time
import pandas as pd
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional, Tuple, Literal

from google.colab import drive, userdata
from google.colab.userdata import SecretNotFoundError, NotebookAccessError
from openai import OpenAI

In [ ]:
# @title 1.2. Configurações e chave OpenAI

def load_openai_key_from_colab(key_name:str):
  try:
      key = userdata.get(key_name)
      os.environ[key_name] = key
      print(f"✅ {key_name} carregada com sucesso a partir do Colab Secrets.")
  except (SecretNotFoundError, NotebookAccessError):
      print(f"⚠️ {key_name} não configurada corretamente no Colab Secrets.")

@dataclass
class AppConfig:
    # Modelo usado para PLANEJAR a consulta (gera JSON de plano)
    plan_model: str = "gpt-4o-mini"
    # Modelo usado para REDIGIR/SUMARIZAR a resposta final
    gen_model: str = "gpt-4o-mini"
    temperature: float = 0.0
    # Controle de contexto e debug
    max_rows_in_context: int = 50
    debug_print_context: bool = True
    debug_print_plan: bool = True

class AppState:
    def __init__(self, config: AppConfig):
        self.config = config
        load_openai_key_from_colab("OPENAI_API_KEY")
        self.client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

        # “Base” estruturada (substitui vector_store/documents)
        self.df_matriz: Optional[pd.DataFrame] = None
        self.df_horarios: Optional[pd.DataFrame] = None
        self.df_calendario: Optional[pd.DataFrame] = None

In [ ]:
# @title 1.3. Integração com Google Drive

def load_json_files_from_drive(drive_root: str, relative_path: str) -> List[Dict]:
    """
    Monta o drive e carrega todos os arquivos .json de uma pasta específica.
    """
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

    # Caminho absoluto baseado na pasta informada
    full_path = f"{drive_root}/{relative_path}"

    if not os.path.isdir(full_path):
        print(f"Erro: A pasta {full_path} não foi encontrada no seu Drive.")
        return []

    all_json_data = []
    print(f"Lendo arquivos de: {full_path}...")

    for filename in os.listdir(full_path):
        if filename.endswith(".json"):
            file_path = os.path.join(full_path, filename)
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = json.load(f)
                    all_json_data.append({
                        "metadata": {"file_name": filename},
                        "content": content
                    })
                print(f"✓ Carregado: {filename}")
            except Exception as e:
                print(f"✗ Erro ao carregar {filename}: {e}")

    return all_json_data

In [ ]:
# @title 2. MetadataRegistry (com persistência no Drive)

# =========================
# 2.1 Modelos de metadados
# =========================
@dataclass
class DatasetMetadata:
    name: str                      # ex.: "horarios"
    description: str               # descrição curta e objetiva
    schema: Dict[str, str]         # campo -> descrição/uso
    use_when: List[str]            # bullets: quando usar
    examples: List[str]            # exemplos de perguntas
    profile: Dict[str, Any]        # resumo inferido dos dados (pequeno)

@dataclass
class RegistryMetadata:
    version: str
    course_default: str
    datasets: Dict[str, DatasetMetadata]


# =========================
# 2.2 Registry (cria, salva, carrega)
# =========================

class MetadataRegistry:
    """
    Registry de metadados (descrição + esquema + perfil dos dados).
    Persistência em JSON no Google Drive para evitar recomputar.
    """

    def __init__(self, version: str = "1.0", course_default: str = "DSM"):
        self.version = version
        self.course_default = course_default
        self.datasets: Dict[str, DatasetMetadata] = {}

    # ---------- caminhos ----------
    @staticmethod
    def _metadata_path(folder_path: str) -> str:
        base = os.path.join(folder_path, "base_metadata")
        os.makedirs(base, exist_ok=True)
        return os.path.join(base, "metadata_registry.json")

    # ---------- criação ----------
    def build_from_dataframes(self, df_horarios, df_matriz, df_calendario, course_default: Optional[str] = None):
        """
        Constrói metadados a partir dos DataFrames já carregados (não reprocessa JSON).
        """
        if course_default:
            self.course_default = course_default

        self.datasets = {
            "horarios": self._build_horarios(df_horarios),
            "matriz": self._build_matriz(df_matriz),
            "calendario": self._build_calendario(df_calendario),
        }

    def _build_horarios(self, df) -> DatasetMetadata:
        schema = {
            "curso": "Sigla do curso (ex.: DSM).",
            "semestre": "Inteiro (1..6).",
            "disciplina": "Nome da disciplina (texto).",
            "professor": "Nome do professor (texto).",
            "dia": "Dia abreviado (Seg/Ter/Qua/Qui/Sex/Sáb).",
            "inicio": "Horário de início (HH:MM).",
            "fim": "Horário de término (HH:MM).",
            "raw_horario": "String original (ex.: 'Seg 18:45-22:15').",
            "source": "Arquivo JSON de origem.",
        }

        profile = {}
        if df is not None and not df.empty:
            profile = {
                "row_count": int(len(df)),
                "semestres_disponiveis": sorted([int(x) for x in df["semestre"].dropna().unique().tolist()]),
                "dias_disponiveis": sorted([str(x) for x in df["dia"].dropna().unique().tolist()]),
                "amostra_professores": sorted([str(x) for x in df["professor"].dropna().unique().tolist()])[:20],
                "amostra_disciplinas": sorted([str(x) for x in df["disciplina"].dropna().unique().tolist()])[:20],
            }

        return DatasetMetadata(
            name="horarios",
            description=(
                "Horários das disciplinas e professores do curso ao longo da semana. "
                "Cada linha representa um encontro (dia + faixa de horário) associado a disciplina e professor."
            ),
            schema=schema,
            use_when=[
                "Perguntas sobre horário de disciplinas (quando/que dia ocorre).",
                "Perguntas sobre horário de professores (em quais dias/horários o professor leciona).",
                "Perguntas que relacionam professor ↔ disciplina ↔ dia/horário.",
                "Listagens e contagens baseadas na grade semanal (ex.: total de encontros).",
            ],
            examples=[
                "Quais são os professores que dão aula na segunda-feira?",
                "Quais são os dias de aula do professor Arley?",
                "Qual é o horário da disciplina Desenvolvimento Web II?",
            ],
            profile=profile
        )

    def _build_matriz(self, df) -> DatasetMetadata:
        schema = {
            "curso": "Sigla do curso (ex.: DSM).",
            "semestre": "Inteiro (1..6).",
            "disciplina": "Nome da disciplina (texto).",
            "carga_horaria": "Carga horária em horas (inteiro).",
            "modalidade": "Modalidade (ex.: presencial/remota).",
            "source": "Arquivo JSON de origem.",
        }

        profile = {}
        if df is not None and not df.empty:
            # modalidades presentes
            mods = sorted([str(x) for x in df["modalidade"].dropna().unique().tolist()])
            profile = {
                "row_count": int(len(df)),
                "semestres_disponiveis": sorted([int(x) for x in df["semestre"].dropna().unique().tolist()]),
                "modalidades_disponiveis": mods,
                "amostra_disciplinas": sorted([str(x) for x in df["disciplina"].dropna().unique().tolist()])[:30],
                "carga_horaria_total_curso": int(df["carga_horaria"].fillna(0).sum()),
            }

        return DatasetMetadata(
            name="matriz",
            description=(
                "Matriz curricular do curso: disciplinas por semestre, com carga horária e modalidade "
                "(presencial/remota). Não contém professor nem horários."
            ),
            schema=schema,
            use_when=[
                "Perguntas sobre carga horária (por semestre, por curso, por modalidade).",
                "Perguntas sobre quais disciplinas existem em um semestre (lista curricular).",
                "Perguntas que exigem soma/contagem de horas (agregações).",
                "Perguntas sobre modalidade (remota/presencial).",
            ],
            examples=[
                "Qual é a carga horária do 5º semestre?",
                "Qual é a carga horária remota do curso?",
                "Quais disciplinas do 3º semestre são remotas?",
            ],
            profile=profile
        )

    def _build_calendario(self, df) -> DatasetMetadata:
        schema = {
            "ano": "Ano do calendário (ex.: 2026).",
            "evento": "Descrição do evento (texto).",
            "inicio": "Data de início (YYYY-MM-DD).",
            "fim": "Data de fim (YYYY-MM-DD).",
            "source": "Arquivo JSON de origem.",
        }

        profile = {}
        if df is not None and not df.empty:
            anos = sorted([int(x) for x in df["ano"].dropna().unique().tolist() if str(x).isdigit()])
            profile = {
                "row_count": int(len(df)),
                "anos_disponiveis": anos,
                "amostra_eventos": sorted([str(x) for x in df["evento"].dropna().unique().tolist()])[:30],
            }

        return DatasetMetadata(
            name="calendario",
            description="Calendário acadêmico: eventos e períodos com datas de início e fim.",
            schema=schema,
            use_when=[
                "Perguntas sobre datas, períodos, feriados e eventos acadêmicos.",
                "Perguntas do tipo 'quando não haverá aula' ou 'quando inicia/termina X'.",
            ],
            examples=[
                "Quando é o Dia do Professor e haverá aula?",
                "Quais são os períodos de rematrícula?",
                "Quando começam as aulas em 2026?",
            ],
            profile=profile
        )

    # ---------- serialização ----------
    def to_dict(self) -> Dict[str, Any]:
        return {
            "version": self.version,
            "course_default": self.course_default,
            "datasets": {k: asdict(v) for k, v in self.datasets.items()}
        }

    @staticmethod
    def from_dict(d: Dict[str, Any]) -> "MetadataRegistry":
        reg = MetadataRegistry(version=d.get("version", "1.0"), course_default=d.get("course_default", "DSM"))
        datasets = d.get("datasets", {})
        for k, v in datasets.items():
            reg.datasets[k] = DatasetMetadata(**v)
        return reg

    # ---------- persistência ----------
    def save_to_drive(self, folder_path: str) -> str:
        path = self._metadata_path(folder_path)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(self.to_dict(), f, ensure_ascii=False, indent=2)
        print(f"✓ Metadados salvos em: {path}")
        return path

    @staticmethod
    def load_from_drive(folder_path: str) -> Optional["MetadataRegistry"]:
        path = MetadataRegistry._metadata_path(folder_path)
        if not os.path.exists(path):
            return None
        with open(path, "r", encoding="utf-8") as f:
            d = json.load(f)
        reg = MetadataRegistry.from_dict(d)
        print(f"✓ Metadados carregados do Drive (cache): {path}")
        return reg


# =========================
# 2.3 Persistência das Tabelas no Drive (cache)
# =========================
def _cache_paths_tables(folder_path: str) -> Dict[str, str]:
    """
    Define os caminhos do cache das tabelas no Drive.
    """
    base = os.path.join(folder_path, "base_structured")
    os.makedirs(base, exist_ok=True)
    return {
        "matriz": os.path.join(base, "df_matriz.parquet"),
        "horarios": os.path.join(base, "df_horarios.parquet"),
        "calendario": os.path.join(base, "df_calendario.parquet"),
    }

def save_tables_to_drive(state: AppState, folder_path: str) -> None:
    """
    Salva as tabelas estruturadas em Parquet no Drive.
    """
    paths = _cache_paths_tables(folder_path)

    if state.df_matriz is None or state.df_horarios is None or state.df_calendario is None:
        raise ValueError("DataFrames não estão prontos para salvar (algum df_* está None).")

    state.df_matriz.to_parquet(paths["matriz"], index=False)
    state.df_horarios.to_parquet(paths["horarios"], index=False)
    state.df_calendario.to_parquet(paths["calendario"], index=False)

    print(f"✓ Tabelas salvas em: {os.path.dirname(paths['matriz'])}")

def load_tables_from_drive(state: AppState, folder_path: str) -> bool:
    """
    Carrega as tabelas estruturadas do cache no Drive.
    Retorna True se carregou; False se cache não existe.
    """
    paths = _cache_paths_tables(folder_path)

    if not (os.path.exists(paths["matriz"]) and os.path.exists(paths["horarios"]) and os.path.exists(paths["calendario"])):
        return False

    state.df_matriz = pd.read_parquet(paths["matriz"])
    state.df_horarios = pd.read_parquet(paths["horarios"])
    state.df_calendario = pd.read_parquet(paths["calendario"])

    print("✓ Tabelas carregadas do Drive (cache).")
    return True


In [ ]:
# @title 3. LLMPlanner (plano em JSON + validação com jsonschema + logs no Drive)

# =========================
# 3.1 JSON Schema do Plano
# =========================

PLAN_SCHEMA: Dict[str, Any] = {
    "type": "object",
    "additionalProperties": False,
    "required": ["dataset", "filters", "select", "group_by", "metrics", "sort", "limit"],
    "properties": {
        "dataset": {
            "type": "string",
            "enum": ["horarios", "matriz", "calendario"]
        },
        "filters": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": ["field", "op", "value"],
                "properties": {
                    "field": {"type": "string"},
                    "op": {
                        "type": "string",
                        "enum": ["=", "!=", "contains", "in", ">=", "<=", ">", "<"]
                    },
                    "value": {}
                }
            }
        },
        "select": {
            "type": "array",
            "items": {"type": "string"}
        },
        "group_by": {
            "type": "array",
            "items": {"type": "string"}
        },
        "metrics": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": ["name", "field", "as"],
                "properties": {
                    "name": {
                        "type": "string",
                        "enum": ["sum", "count", "count_distinct", "min", "max"]
                    },
                    "field": {"type": "string"},
                    "as": {"type": "string"}
                }
            }
        },
        "sort": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": ["field", "dir"],
                "properties": {
                    "field": {"type": "string"},
                    "dir": {"type": "string", "enum": ["asc", "desc"]}
                }
            }
        },
        "limit": {"type": "integer", "minimum": 1, "maximum": 2000}
    }
}


# =========================
# 3.2 Helper: campos permitidos por dataset
# =========================

ALLOWED_FIELDS: Dict[str, List[str]] = {
    "horarios": ["curso", "semestre", "disciplina", "professor", "dia", "inicio", "fim", "raw_horario", "source"],
    "matriz": ["curso", "semestre", "disciplina", "carga_horaria", "modalidade", "source"],
    "calendario": ["ano", "evento", "inicio", "fim", "source"],
}


def _sanitize_plan(plan: Dict[str, Any]) -> Dict[str, Any]:
    """
    Saneamento mínimo:
    - remove campos não permitidos em select/group_by/sort/filters/metrics
    - aplica limit default se ausente
    - garante listas presentes
    """
    dataset = plan.get("dataset")
    allowed = set(ALLOWED_FIELDS.get(dataset, []))

    # defaults
    plan.setdefault("filters", [])
    plan.setdefault("select", [])
    plan.setdefault("group_by", [])
    plan.setdefault("metrics", [])
    plan.setdefault("sort", [])
    plan.setdefault("limit", 200)

    # filtros: remove filtros com field inválido
    clean_filters = []
    for f in plan["filters"]:
        if f.get("field") in allowed:
            clean_filters.append(f)
    plan["filters"] = clean_filters

    # select/group_by: remove campos inválidos e dup
    plan["select"] = sorted(set([c for c in plan["select"] if c in allowed]))
    plan["group_by"] = [c for c in plan["group_by"] if c in allowed]

    # metrics: apenas se field válido
    clean_metrics = []
    for m in plan["metrics"]:
        if m.get("field") in allowed and isinstance(m.get("as"), str) and m.get("as").strip():
            clean_metrics.append(m)
    plan["metrics"] = clean_metrics

    # sort: apenas se field válido
    clean_sort = []
    for s in plan["sort"]:
        if s.get("field") in allowed and s.get("dir") in ("asc", "desc"):
            clean_sort.append(s)
    plan["sort"] = clean_sort

    # limit bounds
    if not isinstance(plan["limit"], int):
        plan["limit"] = 200
    plan["limit"] = max(1, min(2000, plan["limit"]))

    return plan


def _extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    """
    Extrai o primeiro objeto JSON { ... } de um texto.
    """
    if not text:
        return None
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None
    candidate = text[start:end+1]
    try:
        return json.loads(candidate)
    except Exception:
        return None


def jsons_to_tables(json_data: List[Dict]) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Converte os JSONs institucionais em DataFrames estruturados:
    - df_matriz
    - df_horarios
    - df_calendario
    """

    matriz_rows = []
    horarios_rows = []
    calendario_rows = []

    for item in json_data:
        source = item.get("metadata", {}).get("file_name", "desconhecido")
        content = item.get("content", {})

        curso = content.get("sigla") or content.get("curso") or "DSM"

        # -------------------------
        # MATRIZ CURRICULAR
        # -------------------------
        if "matriz" in content:
            for sem in content["matriz"]:
                semestre = sem.get("semestre")
                for disc in sem.get("disciplinas", []):
                    matriz_rows.append({
                        "curso": curso,
                        "semestre": int(semestre),
                        "disciplina": disc.get("nome"),
                        "carga_horaria": int(disc.get("carga_horaria", 0)),
                        "modalidade": disc.get("modalidade"),
                        "source": source
                    })

        # -------------------------
        # HORÁRIOS
        # -------------------------
        elif "semestres" in content:
            for sem in content["semestres"]:
                semestre = sem.get("semestre")
                for disc in sem.get("disciplinas", []):
                    professor = disc.get("professor")
                    disciplina = disc.get("disciplina")
                    for h in disc.get("horarios", []):
                        # exemplo: "Seg 18:45-22:15"
                        try:
                            dia, faixa = h.split(" ", 1)
                            inicio, fim = faixa.split("-")
                        except ValueError:
                            dia, inicio, fim = None, None, None

                        horarios_rows.append({
                            "curso": curso,
                            "semestre": int(semestre),
                            "disciplina": disciplina,
                            "professor": professor,
                            "dia": dia,
                            "inicio": inicio,
                            "fim": fim,
                            "raw_horario": h,
                            "source": source
                        })

        # -------------------------
        # CALENDÁRIO
        # -------------------------
        elif "eventos" in content:
            ano = content.get("ano")
            for ev in content.get("eventos", []):
                calendario_rows.append({
                    "ano": int(ano) if ano else None,
                    "evento": ev.get("nome"),
                    "inicio": ev.get("inicio"),
                    "fim": ev.get("fim"),
                    "source": source
                })

    df_matriz = pd.DataFrame(matriz_rows)
    df_horarios = pd.DataFrame(horarios_rows)
    df_calendario = pd.DataFrame(calendario_rows)

    return df_matriz, df_horarios, df_calendario


# =========================
# 3.3 Persistência opcional de logs do planner
# =========================

def _planner_logs_dir(folder_path: str) -> str:
    d = os.path.join(folder_path, "base_logs", "planner")
    os.makedirs(d, exist_ok=True)
    return d


def save_planner_log(folder_path: str, payload: Dict[str, Any]) -> str:
    """
    Salva um log JSON por execução do planner.
    """
    d = _planner_logs_dir(folder_path)
    ts = time.strftime("%Y%m%d-%H%M%S")
    path = os.path.join(d, f"planner_{ts}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    return path


# =========================
# 3.4 Classe LLMPlanner
# =========================

class LLMPlanner:
    """
    Gera um plano de consulta em JSON com base na pergunta e no MetadataRegistry.
    Valida o plano com jsonschema e aplica saneamento mínimo.
    """

    def __init__(self, client: OpenAI, config: AppConfig):
        self.client = client
        self.config = config

    def plan(self, question: str, registry: MetadataRegistry, drive_folder_path: Optional[str] = None) -> Dict[str, Any]:
        """
        Retorna um dict com o plano validado e saneado.
        Se não conseguir gerar um plano válido, retorna um plano default (dataset mais provável + filtros vazios).
        """
        registry_dict = registry.to_dict()

        system_prompt = (
            "Você é um planejador de consultas para uma base de dados tabular (DataFrames) da Fatec Jacareí.\n"
            "Sua tarefa é produzir APENAS um objeto JSON válido seguindo o schema informado.\n"
            "Regras:\n"
            "- Retorne somente JSON, sem texto adicional.\n"
            "- Escolha dataset em: horarios, matriz, calendario.\n"
            "- Use os campos do schema do dataset escolhido.\n"
            "- Se a pergunta pedir agregação (total/soma/quantas), use 'metrics' e 'group_by' quando necessário.\n"
            "- Defina 'select' com colunas úteis para responder.\n"
            "- Sempre inclua 'limit' (<= 500 se não tiver certeza).\n"
        )

        user_prompt = {
            "question": question,
            "datasets": registry_dict["datasets"],  # inclui description/schema/use_when/examples/profile
            "schema": PLAN_SCHEMA,
            "field_whitelists": ALLOWED_FIELDS,
            "output_instruction": "Gere um JSON conforme o schema, sem nenhum texto extra."
        }

        raw_text = ""
        plan_obj = None
        schema_error = None

        try:
            resp = self.client.chat.completions.create(
                model=self.config.plan_model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": json.dumps(user_prompt, ensure_ascii=False)}
                ],
                temperature=0.0
            )
            raw_text = resp.choices[0].message.content or ""
            plan_obj = _extract_json_object(raw_text)
            if plan_obj is None:
                raise ValueError("Não foi possível extrair JSON do retorno do LLM.")

            # valida schema
            jsonschema.validate(instance=plan_obj, schema=PLAN_SCHEMA)

            # saneamento de campos permitidos
            plan_obj = _sanitize_plan(plan_obj)

        except Exception as e:
            schema_error = str(e)
            # fallback: plano default conservador
            plan_obj = {
                "dataset": "horarios",
                "filters": [],
                "select": ["semestre", "disciplina", "professor", "dia", "inicio", "fim"],
                "group_by": [],
                "metrics": [],
                "sort": [{"field": "semestre", "dir": "asc"}],
                "limit": 200
            }

        # debug/log
        if self.config.debug_print_plan:
            print("=== PLANO GERADO (VALIDADO/SANEADO) ===")
            print(json.dumps(plan_obj, ensure_ascii=False, indent=2))

        if drive_folder_path:
            log_payload = {
                "question": question,
                "raw_text": raw_text,
                "plan": plan_obj,
                "error": schema_error,
                "plan_model": self.config.plan_model,
                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            }
            path = save_planner_log(drive_folder_path, log_payload)
            if self.config.debug_print_plan:
                print(f"✓ Log do planner salvo em: {path}")

        return plan_obj

In [ ]:
# @title 4. QueryExecutor (executa plano do LLM em DataFrames + logs no Drive)

# =========================
# 4.1 Helpers: logs e utilidades
# =========================

def _executor_logs_dir(folder_path: str) -> str:
    d = os.path.join(folder_path, "base_logs", "executor")
    os.makedirs(d, exist_ok=True)
    return d

def save_executor_log(folder_path: str, payload: Dict[str, Any]) -> str:
    d = _executor_logs_dir(folder_path)
    ts = time.strftime("%Y%m%d-%H%M%S")
    path = os.path.join(d, f"executor_{ts}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    return path


def _safe_head(df: pd.DataFrame, n: int = 50) -> pd.DataFrame:
    if df is None:
        return pd.DataFrame()
    return df.head(n)


def _as_compact_text(df: pd.DataFrame, max_rows: int = 50) -> str:
    if df is None or df.empty:
        return "(vazio)"
    return _safe_head(df, max_rows).to_string(index=False)


# =========================
# 4.2 Mapeamento: dataset -> dataframe
# =========================

def get_dataset_df(dataset: str, df_horarios: pd.DataFrame, df_matriz: pd.DataFrame, df_calendario: pd.DataFrame) -> pd.DataFrame:
    if dataset == "horarios":
        return df_horarios
    if dataset == "matriz":
        return df_matriz
    if dataset == "calendario":
        return df_calendario
    raise ValueError(f"Dataset inválido: {dataset}")


# =========================
# 4.3 Operadores de filtro
# =========================

def _apply_filter(df: pd.DataFrame, field: str, op: str, value: Any) -> pd.DataFrame:
    # Se coluna não existir, retorna vazio (falha segura)
    if field not in df.columns:
        return df.iloc[0:0]

    s = df[field]

    # Normalizações de segurança
    if op == "contains":
        # contains: força string
        patt = "" if value is None else str(value)
        return df[s.astype(str).str.contains(patt, case=False, na=False)]

    if op == "in":
        # value deve ser lista
        if not isinstance(value, list):
            return df.iloc[0:0]
        return df[s.isin(value)]

    # Comparações numéricas/datas/strings
    if op == "=":
        return df[s == value]
    if op == "!=":
        return df[s != value]

    # Tentativa de comparação ordenada (>=,<=,>,<)
    # Para datas "YYYY-MM-DD", comparação lexicográfica funciona, desde que padrão ISO.
    if op == ">=":
        return df[s >= value]
    if op == "<=":
        return df[s <= value]
    if op == ">":
        return df[s > value]
    if op == "<":
        return df[s < value]

    # op desconhecido: não filtra
    return df


def _apply_filters(df: pd.DataFrame, filters: List[Dict[str, Any]]) -> Tuple[pd.DataFrame, List[Dict[str, Any]]]:
    applied = []
    out = df
    for f in filters:
        field = f.get("field")
        op = f.get("op")
        value = f.get("value")
        before = len(out)
        out = _apply_filter(out, field, op, value)
        after = len(out)
        applied.append({"field": field, "op": op, "value": value, "rows_before": before, "rows_after": after})
        # curto-circuito
        if after == 0:
            break
    return out, applied


# =========================
# 4.4 Agregações
# =========================

def _apply_aggregations(df: pd.DataFrame, group_by: List[str], metrics: List[Dict[str, Any]]) -> pd.DataFrame:
    """
    metrics: [{"name": "sum|count|count_distinct|min|max", "field": "...", "as": "..."}]
    """
    # Se não houver métricas, apenas retorna df (talvez com select)
    if not metrics:
        return df

    # Garante group_by válido
    group_by = [c for c in group_by if c in df.columns]

    agg_dict = {}
    rename_map = {}
    distinct_requests = []

    for m in metrics:
        name = m.get("name")
        field = m.get("field")
        out_name = m.get("as")

        if field not in df.columns:
            continue

        if name == "count_distinct":
            distinct_requests.append((field, out_name))
        elif name == "count":
            agg_dict[out_name] = (field, "count")
        elif name == "sum":
            agg_dict[out_name] = (field, "sum")
        elif name == "min":
            agg_dict[out_name] = (field, "min")
        elif name == "max":
            agg_dict[out_name] = (field, "max")

    if group_by:
        g = df.groupby(group_by, dropna=False)
        if agg_dict:
            out = g.agg(**{k: v for k, v in agg_dict.items()}).reset_index()
        else:
            out = g.size().reset_index(name="count_rows")

        # count_distinct: calcula à parte e faz merge
        for field, out_name in distinct_requests:
            d = g[field].nunique(dropna=True).reset_index(name=out_name)
            out = out.merge(d, on=group_by, how="left")

        return out
    else:
        # Sem group_by: agrega a tabela inteira
        row = {}
        for out_name, (field, fn) in agg_dict.items():
            if fn == "count":
                row[out_name] = int(df[field].count())
            elif fn == "sum":
                row[out_name] = float(df[field].fillna(0).sum())
            elif fn == "min":
                row[out_name] = df[field].min()
            elif fn == "max":
                row[out_name] = df[field].max()

        for field, out_name in distinct_requests:
            row[out_name] = int(df[field].nunique(dropna=True))

        if not row:
            # fallback conservador
            row = {"count_rows": int(len(df))}
        return pd.DataFrame([row])


# =========================
# 4.5 Ordenação, seleção e limit
# =========================

def _apply_select(df: pd.DataFrame, select: List[str]) -> pd.DataFrame:
    if not select:
        return df
    cols = [c for c in select if c in df.columns]
    if not cols:
        return df
    return df[cols]

def _apply_sort(df: pd.DataFrame, sort: List[Dict[str, Any]]) -> pd.DataFrame:
    if df is None or df.empty or not sort:
        return df
    cols = []
    ascending = []
    for s in sort:
        f = s.get("field")
        d = s.get("dir", "asc")
        if f in df.columns:
            cols.append(f)
            ascending.append(d != "desc")
    if not cols:
        return df
    return df.sort_values(cols, ascending=ascending, kind="mergesort")

def _apply_limit(df: pd.DataFrame, limit: int) -> pd.DataFrame:
    if df is None:
        return pd.DataFrame()
    if not isinstance(limit, int) or limit <= 0:
        limit = 200
    return df.head(limit)


# =========================
# 4.6 Classe QueryExecutor
# =========================

class QueryExecutor:
    """
    Executa um plano validado/saneado (vindo do LLMPlanner) em DataFrames.
    Retorna:
      - df_result: DataFrame final (filtrado/agregado)
      - diag: dicionário com diagnóstico do que foi aplicado
    """

    def __init__(self, config: AppConfig):
        self.config = config

    def execute(
        self,
        plan: Dict[str, Any],
        df_horarios: pd.DataFrame,
        df_matriz: pd.DataFrame,
        df_calendario: pd.DataFrame,
        drive_folder_path: Optional[str] = None
    ) -> Tuple[pd.DataFrame, Dict[str, Any]]:

        dataset = plan.get("dataset")
        df_base = get_dataset_df(dataset, df_horarios, df_matriz, df_calendario)
        if df_base is None or df_base.empty:
            diag = {
                "dataset": dataset,
                "base_rows": 0,
                "applied_filters": [],
                "notes": "DataFrame base vazio."
            }
            return pd.DataFrame(), diag

        base_rows = int(len(df_base))

        # 1) filtros
        df_filtered, applied_filters = _apply_filters(df_base, plan.get("filters", []))

        # 2) agregações
        df_agg = _apply_aggregations(df_filtered, plan.get("group_by", []), plan.get("metrics", []))

        # 3) select (após agregação faz mais sentido)
        df_sel = _apply_select(df_agg, plan.get("select", []))

        # 4) sort e limit
        df_sorted = _apply_sort(df_sel, plan.get("sort", []))
        df_final = _apply_limit(df_sorted, int(plan.get("limit", 200)))

        diag = {
            "dataset": dataset,
            "base_rows": base_rows,
            "rows_after_filters": int(len(df_filtered)),
            "rows_after_agg": int(len(df_agg)),
            "rows_after_select": int(len(df_sel)),
            "rows_final": int(len(df_final)),
            "applied_filters": applied_filters,
            "group_by": plan.get("group_by", []),
            "metrics": plan.get("metrics", []),
            "select": plan.get("select", []),
            "sort": plan.get("sort", []),
            "limit": plan.get("limit", 200),
        }

        # debug
        if self.config.debug_print_context:
            print("=== DIAGNÓSTICO DO EXECUTOR ===")
            print(json.dumps(diag, ensure_ascii=False, indent=2))
            print("\n=== RESULTADO ESTRUTURADO (amostra) ===")
            print(_as_compact_text(df_final, max_rows=min(self.config.max_rows_in_context, 50)))

        # log
        if drive_folder_path:
            payload = {
                "plan": plan,
                "diag": diag,
                "preview": _as_compact_text(df_final, max_rows=50),
                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
            }
            path = save_executor_log(drive_folder_path, payload)
            if self.config.debug_print_plan:
                print(f"✓ Log do executor salvo em: {path}")

        return df_final, diag


In [ ]:
# @title 5. LLMSummarizer (geração da resposta final a partir do resultado estruturado)

# =========================
# 5.1 Logs do sumarizador
# =========================

def _summarizer_logs_dir(folder_path: str) -> str:
    d = os.path.join(folder_path, "base_logs", "summarizer")
    os.makedirs(d, exist_ok=True)
    return d

def save_summarizer_log(folder_path: str, payload: Dict[str, Any]) -> str:
    d = _summarizer_logs_dir(folder_path)
    ts = time.strftime("%Y%m%d-%H%M%S")
    path = os.path.join(d, f"summarizer_{ts}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    return path


# =========================
# 5.2 Utilidades
# =========================

def _df_to_text_for_llm(df: pd.DataFrame, max_rows: int = 50) -> str:
    """
    Converte o DataFrame em texto compacto e estável para o LLM.
    """
    if df is None or df.empty:
        return "(tabela vazia)"
    return df.head(max_rows).to_string(index=False)


# =========================
# 5.3 Classe LLMSummarizer
# =========================

class LLMSummarizer:
    """
    Recebe:
      - pergunta original
      - resultado estruturado (DataFrame)
      - diagnóstico da execução
    Produz:
      - resposta textual final em português institucional
    """

    def __init__(self, client: OpenAI, config: AppConfig):
        self.client = client
        self.config = config

    def summarize(
        self,
        question: str,
        df_result: pd.DataFrame,
        diag: Dict[str, Any],
        drive_folder_path: Optional[str] = None
    ) -> str:

        # Caso vazio: resposta direta, sem chamar LLM
        if df_result is None or df_result.empty:
            return "Não consta nos registros oficiais."

        table_text = _df_to_text_for_llm(
            df_result,
            max_rows=self.config.max_rows_in_context
        )

        system_prompt = (
            "Você é o assistente virtual da Secretaria Acadêmica da Fatec Jacareí.\n"
            "Você deve redigir respostas APENAS com base nos dados fornecidos.\n\n"
            "REGRAS OBRIGATÓRIAS:\n"
            "- Não invente dados, nomes, horários, cargas horárias ou totais.\n"
            "- Não extrapole informações que não estejam explicitamente na tabela.\n"
            "- Se a tabela indicar totais (somas, contagens), use exatamente esses valores.\n"
            "- Se houver múltiplas linhas, organize a resposta em lista ou tabela textual.\n"
            "- Use português formal e claro.\n"
            "- Não mencione termos técnicos como 'DataFrame', 'query', 'pipeline' ou 'plano'.\n"
            "- Se a informação não for suficiente para responder completamente, diga isso.\n"
        )

        user_prompt = (
            f"Pergunta do usuário:\n{question}\n\n"
            f"Resultado oficial da consulta:\n{table_text}\n\n"
            "Redija a resposta final ao usuário com base EXCLUSIVA nesses dados."
        )
        raw_text = ""

        try:
            resp = self.client.chat.completions.create(
                model=self.config.gen_model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=self.config.temperature
            )
            raw_text = resp.choices[0].message.content.strip()

        except Exception:
            raw_text = "Ocorreu um erro ao gerar a resposta."

        # Log opcional
        if drive_folder_path:
            payload = {
                "question": question,
                "diag": diag,
                "table_preview": table_text,
                "answer": raw_text,
                "model": self.config.gen_model,
                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            }
            path = save_summarizer_log(drive_folder_path, payload)
            if self.config.debug_print_plan:
                print(f"✓ Log do sumarizador salvo em: {path}")

        return raw_text

In [ ]:
# @title 6. Execução do Fluxo Completo (Pipeline Final)

# ------------------------------------------------------------
# 6.1 Configuração inicial
# ------------------------------------------------------------

PATH_JSONS = "fatec/jacarei/DSM/PLN/atividades/Atividade 4/arquivos"

config = AppConfig()
state = AppState(config)

# ------------------------------------------------------------
# 6.2 Montagem do Google Drive e caminho base
# ------------------------------------------------------------

if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

full_path = f"/content/drive/MyDrive/{PATH_JSONS}"

# ------------------------------------------------------------
# 6.3 Carregar ou construir tabelas estruturadas (cache)
# ------------------------------------------------------------

if not load_tables_from_drive(state, full_path):
    dados_brutos = load_json_files_from_drive(
        "/content/drive/MyDrive/",
        PATH_JSONS
    )
    if not dados_brutos:
        raise RuntimeError("Nenhum arquivo JSON encontrado.")

    state.df_matriz, state.df_horarios, state.df_calendario = jsons_to_tables(dados_brutos)
    save_tables_to_drive(state, full_path)

# ------------------------------------------------------------
# 6.4 Carregar ou construir MetadataRegistry (cache)
# ------------------------------------------------------------

registry = MetadataRegistry.load_from_drive(full_path)
if registry is None:
    registry = MetadataRegistry(version="1.0", course_default="DSM")
    registry.build_from_dataframes(
        state.df_horarios,
        state.df_matriz,
        state.df_calendario,
        course_default="DSM"
    )
    registry.save_to_drive(full_path)


# ------------------------------------------------------------
# 6.5 Instanciar Planner, Executor e Summarizer
# ------------------------------------------------------------

planner = LLMPlanner(state.client, config)
executor = QueryExecutor(config)
summarizer = LLMSummarizer(state.client, config)

# ------------------------------------------------------------
# 6.6 Loop interativo (pipeline completo)
# ------------------------------------------------------------

while True:
    pergunta = input("\nPergunta (ou 'sair'): ").strip()
    if pergunta.lower() == "sair":
        break

    # 1) Planejamento
    plan = planner.plan(
        question=pergunta,
        registry=registry,
        drive_folder_path=full_path
    )

    # 2) Execução determinística
    df_result, diag = executor.execute(
        plan=plan,
        df_horarios=state.df_horarios,
        df_matriz=state.df_matriz,
        df_calendario=state.df_calendario,
        drive_folder_path=full_path
    )

    # 3) Sumarização final (LLM)
    resposta = summarizer.summarize(
        question=pergunta,
        df_result=df_result,
        diag=diag,
        drive_folder_path=full_path
    )

    print(f"\nR: {resposta}")

